# Temporal Holdout Data from 2022 to 2025

In [1]:
import pandas as pd
from datasets import load_dataset

# Load CAFA5 dataset
dataset = load_dataset("wanglab/cafa5", "temporal_holdout_extended_set_2022_2025")

test_df_proteins = dataset['test'].to_pandas()

In [2]:
print(
    f"""
Missing values:
---------------
protein_id: {test_df_proteins['protein_id'].isna().sum()}
go_ids:     {test_df_proteins['go_ids'].isna().sum()}
go_bp:      {test_df_proteins['go_bp'].isna().sum()}
go_mf:      {test_df_proteins['go_mf'].isna().sum()}
go_cc:      {test_df_proteins['go_cc'].isna().sum()}
sequence:   {test_df_proteins['sequence'].isna().sum()}
"""
)


Missing values:
---------------
protein_id: 0
go_ids:     0
go_bp:      7207
go_mf:      11706
go_cc:      10658
sequence:   12534



In [3]:
with open('temporal-holdout-ext-proteins.txt', 'w') as f:
    for acc in test_df_proteins['protein_id']:
        f.write(acc + '\n')

## Downloading 2025-03 TREMBL Data

```
wget https://ftp.ebi.ac.uk/pub/databases/uniprot/previous_releases/release-2025_03/knowledgebase/knowledgebase2025_03.tar.gz

tar -xzf knowledgebase2025_03.tar.gz
pigz -d uniprot_sprot.dat.gz
pigz -d uniprot_trembl.dat.gz
```

In [ ]:
import pandas as pd
import re

def parse_uniprot(file_path, protein_accessions):
    """
    Parse interlabel .dat file and extract metadata for selected protein accessions (AC line).

    Args:
        file_path (str): Path to the .dat file
        protein_accessions (list): List of accession IDs to extract (e.g., ['Q6GZX4', 'P69905'])

    Returns:
        pandas.DataFrame: Metadata for matching proteins
    """
    protein_accessions = set(protein_accessions)  # for fast lookup

    with open(file_path, 'r') as f:
        entries = f.read().split('//\n')

    data = []
    for entry in entries:
        if not entry.strip():
            continue

        # Extract all AC lines
        ac_matches = re.findall(r'^AC\s+(.+)', entry, re.MULTILINE)
        if not ac_matches:
            continue

        # Combine all ACs from possibly multiple AC lines
        entry_accessions = []
        for ac_line in ac_matches:
            entry_accessions.extend([a.strip() for a in ac_line.split(';') if a.strip()])

        # Check if any of the accessions are in our query list
        matched_accessions = [acc for acc in entry_accessions if acc in protein_accessions]
        if not matched_accessions:
            continue

        # For each accession that matched, extract metadata
        for matched_acc in matched_accessions:
            protein_name = None
            function = None
            organism = None
            subcellular_location = None

            # Protein name
            name_match = re.search(r'DE\s+RecName:\s+Full=(.*?);', entry)
            if name_match:
                protein_name = name_match.group(1).strip()

            # Function
            func_match = re.search(
                r'CC\s+-!-\s+FUNCTION:\s*(.*?)(?=\nCC\s+-!-|\nCC\s+-{3,}|$)',
                entry,
                re.DOTALL
            )
            if func_match:
                function = func_match.group(1).strip().replace('\nCC', '').replace('\n', ' ')

            # Organism
            org_match = re.search(r'OS\s+(.+)', entry)
            if org_match:
                organism = org_match.group(1).strip()

            # Subcellular location
            sub_match = re.search(
                r'CC\s+-!-\s+SUBCELLULAR LOCATION:\s*(.*?)(?=\nCC\s+-!-|\nCC\s+-{3,}|$)',
                entry,
                re.DOTALL
            )
            if sub_match:
                subcellular_location = sub_match.group(1).strip().replace('\nCC', '').replace('\n', ' ')

            data.append({
                'Accession': matched_acc,
                'Protein Name': protein_name,
                'protein_function': function,
                'organism': organism,
                'subcellular_location': subcellular_location
            })

    return pd.DataFrame(data)


# Example usage
if __name__ == "__main__":
    file_path = "uniprot/uniprot_sprot.dat"
    accessions_path = "temporal-holdout-ext-proteins.txt"

    print(f"Reading accessions from {accessions_path}")
    protein_accessions = list(pd.read_csv(accessions_path, header=None)[0])  # your query list
    print(f"Found {len(protein_accessions)} accessions")
    print(f"Parsing uniprot data from {file_path}")
    df = parse_uniprot(file_path, protein_accessions)
    print(f"Done. Found {len(df)} entries.")
    df.to_csv("temporal-holdout-ext-proteins-parsed.csv", index=False)

In [4]:
swissprot_proteins = pd.read_csv('temporal-holdout-ext-proteins-parsed.csv')

In [5]:
swissprot_proteins

,Accession,Protein Name,protein_function,organism,subcellular_location,sequence
0,P0DO85,Strychnine-10-hydroxylase {ECO:0000303|PubMed:...,Monooxygenase involved in the biosynthesis of ...,Strychnos nux-vomica (Poison nut) (Strychnine ...,Membrane {ECO:0000255}; Single-pass membrane p...,MEFSLLYIHTAILGLISLFLILHFVFWRLKSAKGGSAKNSLPPEAG...
1,P0DO87,Strychnine-11-hydroxylase {ECO:0000303|PubMed:...,Monooxygenase involved in the biosynthesis of ...,Strychnos nux-vomica (Poison nut) (Strychnine ...,Membrane {ECO:0000255}; Single-pass membrane p...,MHSAMSFLLLFSLCFLIHCFVFLLIKKKKAKTMDAKTVPPGPKKLP...
2,P46077,14-3-3-like protein GF14 phi,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress).,Nucleus {ECO:0000250|UniProtKB:P48349}. Cytopl...,MAAPPASSSAREEFVYLAKLAEQAERYEEMVEFMEKVAEAVDKDEL...
3,P48348,14-3-3-like protein G-BOX factor 14 kappa {ECO...,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress).,Nucleus {ECO:0000250|UniProtKB:P48349}. Cytopl...,MATTLSRDQYVYMAKLAEQAERYEEMVQFMEQLVSGATPAGELTVE...
4,Q96299,14-3-3-like protein GF14 mu,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress).,Nucleus {ECO:0000250|UniProtKB:P48349}. Cytopl...,MGSGKERDTFVYLAKLSEQAERYEEMVESMKSVAKLNVDLTVEERN...
...,...,...,...,...,...,...
6610,P0DTL6,Zinc finger TRAF-type-containing protein 1 {EC...,NaN,Homo sapiens (Human).,Cytoplasm {ECO:0000250|UniProtKB:Q9QXA1}. Cyto...,MSGAEEAGGGGPAAGPAGSVPAGVGVGVGAGPGAAAGQAAAAALGE...
6611,Q9CQU5,Outer kinetochore KNL1 complex subunit ZWINT {...,Acts as a component of the outer kinetochore K...,Mus musculus (Mouse).,Nucleus {ECO:0000250|UniProtKB:O95229}. Chromo...,MADAEKNAVAEKNNAVATKEVLAEAAAILEPVGLQEEAELPAKIME...
6612,Q9SVY1,Zinc finger protein WIP2,Transcriptional regulator required for normal ...,Arabidopsis thaliana (Mouse-ear cress).,"Nucleus {ECO:0000269|PubMed:20541552, ECO:0000...",MTDPYSNFFTDWFKSNPFHHYPNSSTNPSPHPLPPVTPPSSFFFFP...
6613,Q5TYQ1,Protein zyg-11 homolog,Serves as substrate adapter subunit in an E3 u...,Danio rerio (Zebrafish) (Brachydanio rerio).,NaN,MIHLSQTMDKSSLPSLSDLCMSLVSSRLELFCEMRDDGSLSFREPL...


## Parsing Trembl Data to get remaining proteins metadata 2025-03

```
wget https://ftp.ebi.ac.uk/pub/databases/uniprot/previous_releases/release-2025_03/knowledgebase/knowledgebase2025_03.tar.gz

tar -xzf knowledgebase2025_03.tar.gz
pigz -d uniprot_sprot.dat.gz
pigz -d uniprot_trembl.dat.gz
```

trembl uncompressed is ~850 gb so chunk it to make parsing it in this next step more efficient

In [ ]:
import os

input_file = "uniprot_trembl.dat"
output_dir = os.path.join("data", "trembl_chunks")
os.makedirs(output_dir, exist_ok=True)

chunk_size = 10000  # entries per file
entry_count = 0
chunk_index = 0

outfile = open(f"{output_dir}/chunk_{chunk_index:05d}.dat", "w")

with open(input_file, "r") as f:
    buffer = []
    for line in f:
        buffer.append(line)
        if line.strip() == "//":
            entry_count += 1
            outfile.writelines(buffer)
            buffer = []

            if entry_count % chunk_size == 0:
                outfile.close()
                chunk_index += 1
                outfile = open(f"{output_dir}/chunk_{chunk_index:05d}.dat", "w")

# write any remaining entries
if buffer:
    outfile.writelines(buffer)
outfile.close()

Search those chunks with an sbatch array job

In [ ]:
import sys, os, re, pandas as pd

# Usage: python parse_trembl_chunk_slurm.py <chunks_dir> <output_dir> <accessions_file> <array_idx> <num_chunks>

if len(sys.argv) < 6:
    print("Usage: python parse_trembl_chunk_slurm.py <chunks_dir> <output_dir> <accessions_file> <array_idx> <num_chunks>")
    sys.exit(1)

chunks_dir, output_dir, accessions_file, array_idx, num_chunks = sys.argv[1:6]
os.makedirs(output_dir, exist_ok=True)

# --- Load the accessions list ---
try:
    with open(accessions_file) as f:
        protein_accessions = set(line.strip() for line in f if line.strip())
except FileNotFoundError:
    print(f"Error: Accessions file not found at {accessions_file}")
    sys.exit(1)

# --- Helper Function for cleaning whitespace ---
def clean_text(text):
    """Replaces sequences of two or more whitespace characters with a single space."""
    if text is None:
        return None
    # Use re.sub to replace any one or more whitespace chars (\s+) with a single space
    return re.sub(r'\s+', ' ', text).strip()

def parse_chunk(file_path, protein_accessions):
    results = []
    with open(file_path, "r", encoding="utf-8") as f:
        entry = []
        for line in f:
            entry.append(line)
            if line.strip() == "//":
                text = "".join(entry)
                entry = []

                # Extract all accessions from AC lines
                acs = re.findall(r"^AC\s+(.+)", text, re.MULTILINE)
                if not acs:
                    continue
                accs = [a.strip() for l in acs for a in l.split(';') if a.strip()]

                # Keep only if any match our target set
                matches = [a for a in accs if a in protein_accessions]
                if not matches:
                    continue

                # --- METADATA EXTRACTION ---
                # 1. Try to extract the canonical Recommended Name (RecName)
                name_match = re.search(r"DE\s+RecName:\s+Full=(.*?);", text)
                
                # 2. If RecName is not found, fallback to SubName
                if not name_match:
                    name_match = re.search(r"DE\s+SubName:\s+Full=(.*?);", text)

                func_match = re.search(r"CC\s+-!-\s+FUNCTION:\s*(.*?)(?=\nCC\s+-!-|\nCC\s+-{3,}|$)", text, re.DOTALL)
                org_match  = re.search(r"OS\s+(.+)", text)
                sub_match  = re.search(r"CC\s+-!-\s+SUBCELLULAR LOCATION:\s*(.*?)(?=\nCC\s+-!-|\nCC\s+-{3,}|$)", text, re.DOTALL)
                
                # --- APPLY CLEANING & FORMATTING ---
                
                # Extract and clean Protein Name
                protein_name = clean_text(name_match.group(1)) if name_match else None
                
                # Extract and clean Function (initial cleanup handles raw line breaks/CC tags)
                func_raw = func_match.group(1).strip().replace("\nCC", "").replace("\n", " ") if func_match else None
                protein_function = clean_text(func_raw)
                
                # Extract and clean Organism
                organism = clean_text(org_match.group(1)) if org_match else None
                
                # Extract and clean Subcellular Location (initial cleanup handles raw line breaks/CC tags)
                sub_raw = sub_match.group(1).strip().replace("\nCC", "").replace("\n", " ") if sub_match else None
                subcellular_location = clean_text(sub_raw)

                for acc in matches:
                    results.append({
                        "Accession": acc,
                        "Protein Name": protein_name,
                        "protein_function": protein_function,
                        "organism": organism,
                        "subcellular_location": subcellular_location
                    })
    return pd.DataFrame(results)

# --- SLURM ARRAY LOGIC ---

# Convert array indices and chunk counts to integers
try:
    start_idx = int(array_idx) * int(num_chunks)
    num_chunks_int = int(num_chunks)
except ValueError:
    print("Error: array_idx and num_chunks must be valid integers.")
    sys.exit(1)

# Assuming the maximum number of chunks is 24824
MAX_CHUNKS = 24824 
end_idx = start_idx + num_chunks_int
if end_idx > MAX_CHUNKS:
    end_idx = MAX_CHUNKS

print(f"Parsing chunks from {start_idx} to {end_idx-1} (inclusive)")

for i in range(start_idx, end_idx):

    # --- Construct the correct chunk filename using zero-padding ---
    # :05d ensures the number is padded with leading zeros to 5 digits (e.g., 00001, 24824)
    chunk_file = os.path.join(chunks_dir, f"chunk_{i:05d}.dat")

    if not os.path.exists(chunk_file):
        print(f"⚠️  Chunk file not found: {chunk_file}. Exiting array job for this index.")
        # We use sys.exit(0) here as this is often desired behavior in Slurm array jobs 
        # when a file doesn't exist (e.g., if array index goes beyond available files).
        sys.exit(0)

    # --- Run parsing ---
    df = parse_chunk(chunk_file, protein_accessions)

    # --- Save results ---
    if not df.empty:
        # Create output filename from chunk name
        out_path = os.path.join(output_dir, f"results_{os.path.basename(chunk_file).replace('.dat', '.csv')}")
        df.to_csv(out_path, index=False)
        print(f"✅ Wrote {len(df)} entries to {out_path}")
    else:
        print(f"⚪ No matches found in {chunk_file}")

In [6]:
!cat uniprot/trembl_logs/* | grep "No matches found in" | wc -l

24763


In [7]:
!ls uniprot/trembl_results/ | wc -l

544


25307 - 24763 = 544

Verified, that I covered all the chunks and I got all the proteins that could be found with this search

In [8]:
import glob

# Match all files that start with 'parse_1279275_' and end with '.out'
files = sorted(glob.glob("uniprot/trembl_results/results_chunk_*.csv"))

# Read and concat all, skipping the header row in each file
df = pd.concat(
    [pd.read_csv(f, sep=",", header=None, skiprows=1) for i, f in enumerate(files)],
    ignore_index=True
)

# (optional) if the first row of later chunks got concatenated as headers, fix them:
df.columns = ['Accession','Protein Name','protein_function','organism','subcellular_location', 'sequence']

print(f"✅ Combined {len(files)} files into a dataframe with {len(df):,} rows.")

✅ Combined 544 files into a dataframe with 11,197 rows.


### Get the Dates Added and Dates Modified for the proteins

In [ ]:
import pandas as pd
import numpy as np
import os
import requests
from time import sleep
from tqdm import tqdm

def fetch_uniprot_dates_large(all_accessions, batch_size=500, output_file="temporal-holdout-ext-dates.tsv"):
    """
    Fetch 'date_created' and 'date_modified' for a list of UniProt protein IDs in batches.
    Handles merged accessions, deleted entries, and missing dates.
    Saves results incrementally after each batch.
    """
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    fields = ["accession", "date_created", "date_modified"]

    # Prepare output file with header
    if not os.path.exists(output_file):
        with open(output_file, "w") as f:
            f.write("Accession\tDateCreated\tDateModified\n")

    # Track which accessions were found in batch search
    found_accessions = set()
    batch_results = {}  # {accession: [date_created, date_modified]} for batch results

    # Break IDs into batches
    batches = [all_accessions[i:i + batch_size] for i in range(0, len(all_accessions), batch_size)]

    # First pass: batch search
    for batch_num, batch in enumerate(tqdm(batches, desc="Processing batches")):
        query = " OR ".join([f"accession:{acc}" for acc in batch])
        params = {
            "query": query,
            "format": "json",
            "fields": ",".join(fields),
            "size": batch_size
        }

        try:
            r = requests.get(base_url, params=params, timeout=30)
            if r.status_code != 200:
                print(f"Failed with status {r.status_code} — retrying after delay")
                sleep(5)
                continue
            r.raise_for_status()
            data = r.json()

            results = []
            returned_accessions = set()
            for entry in data.get("results", []):
                acc = entry.get("primaryAccession")
                returned_accessions.add(acc)
                
                # Handle inactive entries
                if entry.get("entryType") == "Inactive":
                    date_created = np.nan
                    date_modified = "DELETED"
                else:
                    entry_audit = entry.get("entryAudit", {})
                    date_created = entry_audit.get("firstPublicDate", None)
                    date_modified = entry_audit.get("lastAnnotationUpdateDate", None)
                
                results.append([acc, date_created, date_modified])
                batch_results[acc] = [date_created, date_modified]

            # Check which input accessions in this batch were found
            # (either as primaryAccession or need individual lookup)
            for input_acc in batch:
                if input_acc in returned_accessions:
                    found_accessions.add(input_acc)

            # Append batch results to TSV
            df = pd.DataFrame(results, columns=["Accession", "DateCreated", "DateModified"])
            df.to_csv(output_file, sep="\t", index=False, mode="a", header=False)

        except Exception as e:
            print(f"Error on batch {batch_num + 1}: {e}")
            sleep(5)

        sleep(0.1)  # rate-limit buffer

    # Second pass: handle missing accessions (those not found in batch search)
    missing_ids = [acc for acc in all_accessions if acc not in found_accessions]
    
    if missing_ids:
        print(f"\n🔍 Found {len(missing_ids)} missing accessions, fetching individually...")
        merged_acc = {}  # {old_acc: new_acc}
        individual_results = []

        # First pass: fetch individual JSON for missing accessions
        for acc in tqdm(missing_ids, desc="Fetching missing JSON"):
            url = f"https://rest.uniprot.org/uniprotkb/{acc}.json"
            try:
                r = requests.get(url, timeout=20)
                if r.status_code != 200:
                    print(f"⚠️ Failed for {acc}: {r.status_code}")
                    # Mark as not found
                    individual_results.append([acc, np.nan, np.nan])
                    continue
                data = r.json()

                acc_id = data.get("primaryAccession", acc)
                if acc_id != acc:
                    # Store mapping for second pass
                    merged_acc[acc] = acc_id
                    print(f"ℹ️ {acc} was merged into {acc_id}")
                    continue  # Don't add results yet, fetch in second loop

                # Handle normal / inactive entries
                if data.get("entryType") == "Inactive":
                    date_created = np.nan
                    date_modified = "DELETED"
                else:
                    entry_audit = data.get("entryAudit", {})
                    date_created = entry_audit.get("firstPublicDate", np.nan)
                    date_modified = entry_audit.get("lastAnnotationUpdateDate", np.nan)

                individual_results.append([acc, date_created, date_modified])

            except Exception as e:
                print(f"❌ Error for {acc}: {e}")
                individual_results.append([acc, np.nan, np.nan])
            sleep(0.1)

        # Second pass: handle merged accessions
        if merged_acc:
            print(f"\n🔄 Handling {len(merged_acc)} merged accessions...")
            for old_acc, new_acc in tqdm(merged_acc.items(), desc="Fetching merged JSON"):
                url = f"https://rest.uniprot.org/uniprotkb/{new_acc}.json"
                try:
                    r = requests.get(url, timeout=20)
                    if r.status_code != 200:
                        print(f"⚠️ Failed for {new_acc}: {r.status_code}")
                        # Mark original as merged but with no dates
                        individual_results.append([old_acc, np.nan, "MERGED"])
                        continue
                    data = r.json()

                    acc_id = data.get("primaryAccession", new_acc)
                    if acc_id != new_acc:
                        print(f"⚠️ Mismatch: {new_acc} resolved to {acc_id}")

                    if data.get("entryType") == "Inactive":
                        date_created = np.nan
                        date_modified = "DELETED"
                    else:
                        entry_audit = data.get("entryAudit", {})
                        date_created = entry_audit.get("firstPublicDate", np.nan)
                        date_modified = entry_audit.get("lastAnnotationUpdateDate", np.nan)

                    # Mark as merged with dates from new accession
                    individual_results.append([old_acc, date_created, "MERGED"])

                except Exception as e:
                    print(f"❌ Error for {old_acc} → {new_acc}: {e}")
                    individual_results.append([old_acc, np.nan, "MERGED"])
                sleep(0.1)

        # Append individual results to TSV
        if individual_results:
            df_individual = pd.DataFrame(individual_results, columns=["Accession", "DateCreated", "DateModified"])
            df_individual.to_csv(output_file, sep="\t", index=False, mode="a", header=False)

    print(f"✅ Done. Saved to {output_file}")



if __name__ == "__main__":
    accessions_path = "temporal-holdout-ext-proteins.txt"

    print(f"Reading accessions from {accessions_path}")
    protein_accessions = list(pd.read_csv(accessions_path, header=None)[0])  # your query list
    print(f"Found {len(protein_accessions)} accessions")

    fetch_uniprot_dates_large(protein_accessions, batch_size=100)

In [9]:
trembl_test = df
trembl_test

,Accession,Protein Name,protein_function,organism,subcellular_location,sequence
0,Q59VN4,Histone H4 {ECO:0000256|RuleBase:RU000528},Core component of nucleosome. Nucleosomes wrap...,Candida albicans (strain SC5314 / ATCC MYA-287...,Chromosome {ECO:0000256|ARBA:ARBA00004286}. Nu...,MSGTGRGKGGKGLGKGGAKRHRKILRDNIQGITKPAIRRLARRGGV...
1,B6JV95,Histone H3 {ECO:0000256|RuleBase:RU004471},NaN,Schizosaccharomyces japonicus (strain yFS275 /...,Chromosome {ECO:0000256|ARBA:ARBA00004286}. Nu...,MARTKQTARKSTGGKAPRKQLASKAARKSAPPTTGGVKKPHRYRPG...
2,A0A2H0ZI42,Hyphally-regulated cell wall protein N-termina...,GPI-anchored cell wall protein involved in cel...,Candidozyma auris (Yeast) (Candida auris).,Membrane {ECO:0000256|ARBA:ARBA00004589}; Lipi...,MAFNFVRGWLLLAFYLSATWALTITENTVNVGALNIKIGSLTINPG...
3,A0A2H1A1X2,E3 ubiquitin-protein ligase {ECO:0000256|RuleB...,Ubiquitin ligase protein which is a component ...,Candidozyma auris (Yeast) (Candida auris).,NaN,MCQSDQQFLSPQEVFTSIELHTKNRDLDSKKYGFSNSSSVTDTLSE...
4,A0A2H1A7H1,Zn(2)-C6 fungal-type domain-containing protein...,NaN,Candidozyma auris (Yeast) (Candida auris).,NaN,MSMKEEQQQHQQQPQDKKKSSEKKRRFHSKSRSGCSTCKKRRVKCD...
...,...,...,...,...,...,...
11192,A0A6I8RXM5,Mothers against decapentaplegic homolog {ECO:0...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,Cytoplasm {ECO:0000256|RuleBase:RU361195}. Nuc...,MFRTKRSVLVRRLWRSRAPGESGEGEGGERVHGAGGCCLGKSANKV...
11193,F7EN59,Secreted frizzled-related protein 1 {ECO:00002...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,Cell membrane {ECO:0000256|ARBA:ARBA00004651};...,MNSEGGIWPLLLFWVTPGILSQAPQASEYDYVSFQPDLGRQYQSGR...
11194,A0A803J722,PC-esterase domain-containing 1A {ECO:0000313|...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MCIYTASDRSRRDMSCFSTEHAQQLLHNKYVAILGDSIQRSVYKDL...
11195,F6SMM0,"Mago homolog, exon junction complex subunit {E...",NaN,Xenopus tropicalis (Western clawed frog) (Silu...,Nucleus {ECO:0000256|ARBA:ARBA00004123}.,MELEGKLRYANNSNYKNDVMIRKEAYVHKSVMEELKRIIDDSEITK...


In [10]:
temporal_holdout_ext = pd.concat([swissprot_proteins,trembl_test])

In [11]:
temporal_holdout_ext

,Accession,Protein Name,protein_function,organism,subcellular_location,sequence
0,P0DO85,Strychnine-10-hydroxylase {ECO:0000303|PubMed:...,Monooxygenase involved in the biosynthesis of ...,Strychnos nux-vomica (Poison nut) (Strychnine ...,Membrane {ECO:0000255}; Single-pass membrane p...,MEFSLLYIHTAILGLISLFLILHFVFWRLKSAKGGSAKNSLPPEAG...
1,P0DO87,Strychnine-11-hydroxylase {ECO:0000303|PubMed:...,Monooxygenase involved in the biosynthesis of ...,Strychnos nux-vomica (Poison nut) (Strychnine ...,Membrane {ECO:0000255}; Single-pass membrane p...,MHSAMSFLLLFSLCFLIHCFVFLLIKKKKAKTMDAKTVPPGPKKLP...
2,P46077,14-3-3-like protein GF14 phi,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress).,Nucleus {ECO:0000250|UniProtKB:P48349}. Cytopl...,MAAPPASSSAREEFVYLAKLAEQAERYEEMVEFMEKVAEAVDKDEL...
3,P48348,14-3-3-like protein G-BOX factor 14 kappa {ECO...,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress).,Nucleus {ECO:0000250|UniProtKB:P48349}. Cytopl...,MATTLSRDQYVYMAKLAEQAERYEEMVQFMEQLVSGATPAGELTVE...
4,Q96299,14-3-3-like protein GF14 mu,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress).,Nucleus {ECO:0000250|UniProtKB:P48349}. Cytopl...,MGSGKERDTFVYLAKLSEQAERYEEMVESMKSVAKLNVDLTVEERN...
...,...,...,...,...,...,...
11192,A0A6I8RXM5,Mothers against decapentaplegic homolog {ECO:0...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,Cytoplasm {ECO:0000256|RuleBase:RU361195}. Nuc...,MFRTKRSVLVRRLWRSRAPGESGEGEGGERVHGAGGCCLGKSANKV...
11193,F7EN59,Secreted frizzled-related protein 1 {ECO:00002...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,Cell membrane {ECO:0000256|ARBA:ARBA00004651};...,MNSEGGIWPLLLFWVTPGILSQAPQASEYDYVSFQPDLGRQYQSGR...
11194,A0A803J722,PC-esterase domain-containing 1A {ECO:0000313|...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MCIYTASDRSRRDMSCFSTEHAQQLLHNKYVAILGDSIQRSVYKDL...
11195,F6SMM0,"Mago homolog, exon junction complex subunit {E...",NaN,Xenopus tropicalis (Western clawed frog) (Silu...,Nucleus {ECO:0000256|ARBA:ARBA00004123}.,MELEGKLRYANNSNYKNDVMIRKEAYVHKSVMEELKRIIDDSEITK...


In [12]:
temporal_holdout_ext.columns

Index(['Accession', 'Protein Name', 'protein_function', 'organism',
       'subcellular_location', 'sequence'],
      dtype='object')

In [ ]:
temporal_holdout_ext = final_temporal_holdout_ext.rename(columns={'Accession':'protein_id', 'Protein Name':'protein_names'})

In [14]:
test_df_proteins

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,...,go_mf_created,go_cc_created,interpro_ids,structure_path,string_id,interaction_partners,full_interaction_info,interpro_location,date_added,last_modified
0,A0A014NDJ0,None,None,None,NaN,None,None,"[GO:0008150, GO:0035821, GO:0044003, GO:004440...","[GO:0008150, GO:0035821, GO:0044003, GO:004440...","[GO:0003674, GO:0004857, GO:0004866, GO:000486...",...,20250313,None,None,None,None,None,None,None,None,None
1,A0A014NF26,None,None,None,NaN,None,None,"[GO:0008150, GO:0009605, GO:0009607, GO:003582...","[GO:0008150, GO:0009605, GO:0009607, GO:003582...","[GO:0003674, GO:0030545, GO:0030547, GO:009877...",...,20240304,20240304,None,None,None,None,None,None,None,None
2,A0A023GQA5,None,None,None,NaN,None,None,"[GO:0003674, GO:0005488, GO:0005515]",None,"[GO:0003674, GO:0005488, GO:0005515]",...,20250426,None,None,None,None,None,None,None,None,None
3,A0A024B7W1,None,None,None,NaN,None,None,"[GO:0002376, GO:0002682, GO:0002683, GO:000283...","[GO:0002376, GO:0002682, GO:0002683, GO:000283...",None,...,None,20230117,None,None,None,None,None,None,None,None
4,A0A024FSH6,None,None,None,NaN,None,None,"[GO:0000381, GO:0003006, GO:0007275, GO:000753...","[GO:0000381, GO:0003006, GO:0007275, GO:000753...",None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17807,X5JA13,Exocyst complex component SEC10a (AtSec10a) (E...,Component of the exocyst complex involved in t...,Arabidopsis thaliana (Mouse-ear cress),825.0,"Cytoplasm, cytosol {ECO:0000269|PubMed:2430768...",MTEGIRARGPRSSSVNSVPLILDIEDFKGDFSFDALFGNLVNDLLP...,"[GO:0007154, GO:0008037, GO:0008150, GO:000985...","[GO:0007154, GO:0008037, GO:0008150, GO:000985...",None,...,None,None,"[IPR048625, IPR048627, IPR009976]",AF-X5JA13-F1-model_v4.cif,3702.X5JA13,"[ATEXO70G2, EXO70H1, F11F8_10, VPS54, EXO70F1,...",None,"{""IPR009976"": [20, 812], ""IPR048625"": [90, 206...",2017-01-18,2025-06-18
17808,X6RHX1,None,None,None,NaN,None,None,"[GO:0005575, GO:0005622, GO:0005737, GO:000578...",None,None,...,None,20241022,None,None,None,None,None,None,None,None
17809,X6RK18,None,None,None,NaN,None,None,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",None,None,...,None,20241022,None,None,None,None,None,None,None,None
17810,X6RLK1,None,None,None,NaN,None,None,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",None,None,...,None,20241022,None,None,None,None,None,None,None,None


In [15]:
test_df_proteins.columns

Index(['protein_id', 'protein_names', 'protein_function', 'organism', 'length',
       'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc',
       'go_bp_created', 'go_mf_created', 'go_cc_created', 'interpro_ids',
       'structure_path', 'string_id', 'interaction_partners',
       'full_interaction_info', 'interpro_location', 'date_added',
       'last_modified'],
      dtype='object')

In [16]:
test_df_proteins = test_df_proteins.drop(columns=['protein_names', 'protein_function', 'organism', 'length',
       'subcellular_location', 'sequence',  'interpro_ids',
       'structure_path', 'string_id', 'interaction_partners',
       'full_interaction_info', 'interpro_location', 'date_added',
       'last_modified'])

In [17]:
final_temporal_holdout_ext = pd.merge(temporal_holdout_ext, test_df_proteins, on='protein_id')

In [18]:
holdout_ext_dates = pd.read_csv('temporal-holdout-ext-dates.tsv', sep='\t')
holdout_ext_dates = holdout_ext_dates.rename(columns={'Accession':'protein_id', 'DateCreated':'date_added',	'DateModified':'last_modified'})

In [19]:
final_temporal_holdout_ext = pd.merge(final_temporal_holdout_ext, holdout_ext_dates, on='protein_id')

In [20]:
final_temporal_holdout_ext.columns

Index(['protein_id', 'protein_names', 'protein_function', 'organism',
       'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc',
       'go_bp_created', 'go_mf_created', 'go_cc_created', 'date_added',
       'last_modified'],
      dtype='object')

In [21]:
print(
    f"""
Missing values:
---------------
protein_id:           {final_temporal_holdout_ext['protein_id'].isna().sum()}
protein_names:        {final_temporal_holdout_ext['protein_names'].isna().sum()}
protein_function:     {final_temporal_holdout_ext['protein_function'].isna().sum()}
organism:             {final_temporal_holdout_ext['organism'].isna().sum()}
subcellular_location: {final_temporal_holdout_ext['subcellular_location'].isna().sum()}
go_ids:               {final_temporal_holdout_ext['go_ids'].isna().sum()}
go_bp:                {final_temporal_holdout_ext['go_bp'].isna().sum()}
go_mf:                {final_temporal_holdout_ext['go_mf'].isna().sum()}
go_cc:                {final_temporal_holdout_ext['go_cc'].isna().sum()}
sequence:             {final_temporal_holdout_ext['sequence'].isna().sum()}
go_bp_created:        {final_temporal_holdout_ext['go_bp_created'].isna().sum()}
go_mf_created:        {final_temporal_holdout_ext['go_mf_created'].isna().sum()}
go_cc_created:        {final_temporal_holdout_ext['go_cc_created'].isna().sum()}
date_added:           {final_temporal_holdout_ext['date_added'].isna().sum()}
last_modified:        {final_temporal_holdout_ext['last_modified'].isna().sum()}
"""
)


Missing values:
---------------
protein_id:           0
protein_names:        0
protein_function:     9379
organism:             0
subcellular_location: 5342
go_ids:               0
go_bp:                7207
go_mf:                11706
go_cc:                10658
sequence:             0
go_bp_created:        7207
go_mf_created:        11706
go_cc_created:        10658
date_added:           45
last_modified:        0



In [22]:
print(
    f"""
Unique values:
---------------
protein_id:      {final_temporal_holdout_ext['protein_id'].nunique()}
sequence:        {final_temporal_holdout_ext['sequence'].nunique()}
protein_names:   {final_temporal_holdout_ext['protein_names'].nunique()}
"""
)


Unique values:
---------------
protein_id:      17812
sequence:        17616
protein_names:   14270



In [23]:
final_temporal_holdout_ext['protein_names'] = final_temporal_holdout_ext['protein_names'].str.replace(r'\s?\{.*?\}', '', regex=True).str.strip()
final_temporal_holdout_ext['protein_function'] = final_temporal_holdout_ext['protein_function'].str.replace(r'\s?\{.*?\}|\s?\((?:Pub).*?\)', '', regex=True).str.strip()
final_temporal_holdout_ext['organism'] = final_temporal_holdout_ext['organism'].str.replace(r'\.+$', '', regex=True).str.strip()
final_temporal_holdout_ext['subcellular_location'] = final_temporal_holdout_ext['subcellular_location'].str.replace(r'\s?\{.*?\}|\s?\((?:Pub).*?\)', '', regex=True).str.strip()

In [24]:
final_temporal_holdout_ext['protein_names'] = final_temporal_holdout_ext['protein_names'].str.replace(r'\s+', ' ', regex=True).str.strip()
final_temporal_holdout_ext['protein_function'] = final_temporal_holdout_ext['protein_function'].str.replace(r'\s+', ' ', regex=True).str.strip()
final_temporal_holdout_ext['organism'] = final_temporal_holdout_ext['organism'].str.replace(r'\s+', ' ', regex=True).str.strip()
final_temporal_holdout_ext['subcellular_location'] = final_temporal_holdout_ext['subcellular_location'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [25]:
final_temporal_holdout_ext['protein_names'] = final_temporal_holdout_ext['protein_names'].str.replace(r'[.\s]+$', '', regex=True).str.strip()
final_temporal_holdout_ext['protein_function'] = final_temporal_holdout_ext['protein_function'].str.replace(r'[.\s]+$', '', regex=True).str.strip()
final_temporal_holdout_ext['organism'] = final_temporal_holdout_ext['organism'].str.replace(r'[.\s]+$', '', regex=True).str.strip()
final_temporal_holdout_ext['subcellular_location'] = final_temporal_holdout_ext['subcellular_location'].str.replace(r'[.\s]+$', '', regex=True).str.strip()

In [26]:
final_temporal_holdout_ext['length'] = final_temporal_holdout_ext['sequence'].str.len()

In [27]:
final_temporal_holdout_ext.to_csv('temp_final_temporal_holdout_ext.csv', header=True, index=False)

In [28]:
final_temporal_holdout_ext = pd.read_csv('temp_final_temporal_holdout_ext.csv')

In [29]:
from tqdm import tqdm
from goatools.obo_parser import GODag
import pandas as pd
import numpy as np

# Load ontology
print("Loading Gene Ontology structure...")
# Make sure the path is correct for your environment
obodag = GODag("../data/go-basic.obo") 

# Precompute ALL ancestor terms for all GO terms
print("Precomputing ALL ancestor terms...")
go_to_ancestors = {}
for go_id, term in tqdm(obodag.items(), desc="Building ancestor map"):
    
    # --- CORRECTED ANCESTOR LOOKUP for older goatools ---
    
    # 1. Get ALL ancestors (parents, grandparents, etc.) - this returns a SET of GO ID STRINGS,
    #    but does NOT include the term itself in older versions.
    all_ancestors_no_self = term.get_all_parents() 
    
    # 2. Manually create the set that includes the term itself.
    all_ancestors_with_self = set(all_ancestors_no_self)
    all_ancestors_with_self.add(go_id)
    
    # 3. Store the set of all ancestors EXCLUDING the term itself.
    #    This is the set of terms that MUST be present for the annotation to be complete.
    go_to_ancestors[go_id] = all_ancestors_with_self - {go_id}
    
# ----------------------------------------------------------------------
# MODIFIED CHECKING FUNCTION (NO CHANGES NEEDED HERE)
# ----------------------------------------------------------------------

def check_ancestor_completeness(go_terms):
    """
    Checks if a list of GO terms contains ALL of the ancestors for every term in the list.
    
    Returns: 
        True if all ancestors are present, False otherwise.
    """
    if not isinstance(go_terms, (list, set)) or not go_terms:
        return True # Treat empty/invalid input as complete (no missing ancestors)
    
    # 1. Get the set of all unique ancestors required by the original terms
    required_ancestors = set()
    for go_id in go_terms:
        # NOTE: If a GO_ID is invalid (not in obodag), it won't be in the map.
        if go_id in go_to_ancestors:
            required_ancestors.update(go_to_ancestors[go_id])
            
    # 2. The annotation is complete if the required ancestors are a subset of the terms provided.
    terms_set = set(go_terms)
    missing_ancestors = required_ancestors - terms_set
    
    # Return True if nothing is missing, False if ancestors are missing
    return not bool(missing_ancestors)


# ----------------------------------------------------------------------
# APPLY TO DATAFRAME AND COUNT (NO CHANGES NEEDED HERE)
# ----------------------------------------------------------------------

print("\nChecking GO Ancestor Completeness...")
tqdm.pandas(desc="Checking GO Ancestor Completeness")

# For 'go_ids' column
# This applies the check and returns True/False for each row.
completeness_check = final_temporal_holdout_ext['go_ids'].progress_apply(check_ancestor_completeness)

# Count the number of rows where the check returned False (i.e., ancestors are missing)
missing_ancestors_count = (~completeness_check).sum()

print(f"\nTotal rows checked: {len(final_temporal_holdout_ext)}")
print(f"Number of rows in 'go_ids' with MISSING ancestors: {missing_ancestors_count}")
print(f"Percentage of incomplete rows: {(missing_ancestors_count / len(final_temporal_holdout_ext)) * 100:.2f}%")

Loading Gene Ontology structure...
../data/go-basic.obo: fmt(1.2) rel(2023-01-01) 46,739 Terms
Precomputing ALL ancestor terms...


Building ancestor map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 46739/46739 [00:01<00:00, 38559.32it/s]



Checking GO Ancestor Completeness...


Checking GO Ancestor Completeness: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 17812/17812 [00:00<00:00, 887427.16it/s]


Total rows checked: 17812
Number of rows in 'go_ids' with MISSING ancestors: 0
Percentage of incomplete rows: 0.00%


In [30]:
from goatools.obo_parser import GODag

# Load the ontology file (.obo). You can download it from http://purl.obolibrary.org/obo/go.obo
obo_file = "../data/go-basic.obo"
go_dag = GODag(obo_file)

# Build dictionary mapping GO term → depth
go_to_depth = {go_id: term.depth for go_id, term in go_dag.items()}

# List of GO columns to sort
go_columns = ["go_ids", "go_bp", "go_mf", "go_cc"]

# Function to sort a list of GO terms by depth
def sort_by_depth(go_list, go_to_depth):
    if isinstance(go_list, list) and go_list:
        # sort ascending by depth (shallowest first)
        return sorted(go_list, key=lambda go: go_to_depth[go])
    return go_list  # keep None or empty as-is

# Apply to all columns with a progress bar
from tqdm import tqdm
tqdm.pandas(desc="Sorting GO term columns")

for col in go_columns:
    final_temporal_holdout_ext[col] = final_temporal_holdout_ext[col].progress_apply(lambda x: sort_by_depth(x, go_to_depth))

../data/go-basic.obo: fmt(1.2) rel(2023-01-01) 46,739 Terms


Sorting GO term columns: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17812/17812 [00:00<00:00, 913759.09it/s]


In [31]:
import pandas as pd
import numpy as np

def validate_go_columns_fast(df, columns, go_to_depth):
    results = {}
    depth_lookup = go_to_depth  # local copy

    for col in columns:
        def check_sorted(go_list):
            # skip empty or invalid rows
            if not isinstance(go_list, (list, np.ndarray)) or not go_list:
                return True
            depths = [depth_lookup[go] for go in go_list]
            return all(earlier <= later for earlier, later in zip(depths, depths[1:]))

        results[col] = df[col].apply(check_sorted).all()
    return results

columns_to_check = ["go_ids", "go_bp", "go_mf", "go_cc"]
validation_results = validate_go_columns_fast(final_temporal_holdout_ext, columns_to_check, go_to_depth)

print("All GO term columns sorted by depth:")
for col, is_valid in validation_results.items():
    print(f"{col}: {is_valid}")

All GO term columns sorted by depth:
go_ids: True
go_bp: True
go_mf: True
go_cc: True


In [32]:
from tqdm import tqdm
tqdm.pandas(desc="Sanity check: GO categories sum to go_ids")

def check_go_sum(row):
    go_ids = set(row["go_ids"]) if isinstance(row["go_ids"], list) else set()
    go_union = set()
    for col in ["go_bp", "go_mf", "go_cc"]:
        if isinstance(row[col], list):
            go_union.update(row[col])
    return go_ids == go_union

# Apply to all rows
sanity_check_passed = final_temporal_holdout_ext.progress_apply(check_go_sum, axis=1).all()

print(f"Sanity check: All rows have go_bp + go_mf + go_cc == go_ids? {sanity_check_passed}")

# Optional: if you want to see which rows fail
failed_rows = final_temporal_holdout_ext[~final_temporal_holdout_ext.progress_apply(check_go_sum, axis=1)]
print(f"Number of rows failing sanity check: {len(failed_rows)}")

Sanity check: GO categories sum to go_ids: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 17812/17812 [00:00<00:00, 64722.91it/s]


Sanity check: All rows have go_bp + go_mf + go_cc == go_ids? True


Sanity check: GO categories sum to go_ids: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 17812/17812 [00:00<00:00, 65824.89it/s]

Number of rows failing sanity check: 0


# Download AlphaFold Structures for all Proteins

Follow instructions on setting up Google Cloud account and downloading from google cloud https://github.com/google-deepmind/alphafold/blob/main/afdb/README.md

In [ ]:
with open("alphafold_temporal_holdout_ext_manifest.txt", "w") as f:
    for protein in (list(final_temporal_holdout_ext['protein_id'])):
        f.write(f"gs://public-datasets-deepmind-alphafold-v4/AF-{protein}-F1-model_v4.cif\n")

```cat alphafold_temporal_holdout_ext_manifest.txt | gsutil -m cp -I af_db_interlabel/```

**Trying it a second way as only a quarter had structures from the above command**

In [ ]:
import requests
import os
from tqdm import tqdm

# Define the base URL structure
AF_BASE_URL = "https://alphafold.ebi.ac.uk/files/AF-{}-F1-model_v6.pdb"
DOWNLOAD_FOLDER = "alphafold_structures"

def download_alphafold_model(uniprot_accession, local_folder):
    """Constructs the URL and downloads the corresponding AlphaFold PDB file."""
    
    # 1. Construct the full download URL
    url = AF_BASE_URL.format(uniprot_accession)
    
    # 2. Define the local path and filename
    filename = url.split('/')[-1]
    local_path = os.path.join(local_folder, filename)
    
    # Create the folder if it doesn't exist
    os.makedirs(local_folder, exist_ok=True)
    
    print(f"\nAttempting to download {filename}...")
    
    try:
        # Use streaming request to avoid loading the entire file into memory
        with requests.get(url, stream=True) as r:
            r.raise_for_status() # Check for errors (e.g., 404 Not Found)
            
            # Get the total file size for the progress bar
            total_size = int(r.headers.get('content-length', 0))
            block_size = 8192 # Chunk size for streaming
            
            with open(local_path, 'wb') as f:
                with tqdm(
                    total=total_size, 
                    unit='iB', 
                    unit_scale=True, 
                    desc=filename, 
                    ncols=80
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=block_size):
                        f.write(chunk)
                        pbar.update(len(chunk))
        
        print(f"✅ Successfully downloaded to: {local_path}")
        return local_path
        
    except requests.exceptions.HTTPError as e:
        print(f"❌ Error for {uniprot_accession}: File not found or failed to download (HTTP {e.response.status_code}).")
        return None
    except requests.exceptions.RequestException as e:
        print(f"❌ Network error for {uniprot_accession}: {e}")
        return None

# ----------------------------------------------------------------------
# EXAMPLE USAGE
# ----------------------------------------------------------------------

# Your list of UniProt accessions

with open("temporal_holdout_ext_accessions.txt", "r") as f:
    accession_list = [line.strip() for line in f if line.strip()]

# Loop through your list and download
for acc in accession_list:
    download_alphafold_model(acc, DOWNLOAD_FOLDER)

print(f"\nAll downloads attempted. Check the '{DOWNLOAD_FOLDER}' folder.")

### Counting number of structures

In [33]:
import os

# --- Configuration ---
ACCESSION_FILE = "temporal-holdout-ext-proteins.txt"
DIR_ALPHAFOLD = "alphafold_structures" 

# --- Main Logic ---
print("Starting structure check...")

try:
    # 1. Load in the accession file
    with open(ACCESSION_FILE, 'r') as f:
        # Assuming one accession per line, stripping whitespace
        all_accessions = {line.strip() for line in f if line.strip()}
except FileNotFoundError:
    print(f"Error: Accession file '{ACCESSION_FILE}' not found.")
    exit()
except Exception as e:
    print(f"An error occurred while reading the accession file: {e}")
    exit()

total_accessions = len(all_accessions)
accessions_with_structures = set()
structures_found_count = 0

# 2. Check each accession for structures in the target directory
for accession in all_accessions:

    pdb_path = os.path.join(DIR_ALPHAFOLD, f"AF-{accession}-F1-model_v6.pdb")
    if os.path.exists(pdb_path):
        structures_found_count += 1
        accessions_with_structures.add(accession)
        
# --- Output Results ---
print("\n--- Structure Existence Report ---")
print(f"Total accessions loaded from '{ACCESSION_FILE}': {total_accessions}")
print(f"Directory checked: '{DIR_ALPHAFOLD}'")
print("-" * 40)

# Calculate unique accessions with structures and those without
unique_accessions_with_structures = len(accessions_with_structures)
accessions_without_structures = total_accessions - unique_accessions_with_structures

print(f"Accessions with a structure found: {unique_accessions_with_structures}")
print(f"Accessions with NO structure found: {accessions_without_structures}")
print("-" * 40)


Starting structure check...

--- Structure Existence Report ---
Total accessions loaded from 'temporal-holdout-ext-proteins.txt': 17812
Directory checked: 'alphafold_structures'
----------------------------------------
Accessions with a structure found: 16081
Accessions with NO structure found: 1731
----------------------------------------


In [34]:
[c for c in all_accessions if c not in accessions_with_structures][:5]

['Q6RX08', 'A0A8M9QGF1', 'A0A8J1JF89', 'E7F1J4', 'A0A8J1JJY2']

Switch to Bash terminal

In [ ]:
mkdir -p af_temp_holdout_2022_2025_shards

i=0
shard=0

for file in alphafold_structures/*.pdb; do
    # Create a new folder every 5000 files
    if (( i % 5000 == 0 )); then
        shard_dir="af_temp_holdout_2022_2025_shards/shard_${shard}"
        mkdir -p "$shard_dir"
        ((shard++))
    fi

    cp "$file" "$shard_dir/"
    ((i++))
done

In [ ]:
cd af_temp_holdout_2022_2025_shards

for dir in shard_*; do
    tar -czf "${dir}.tar.gz" "$dir"
    rm -r "$dir"  # optional cleanup
done

cd ..

In [ ]:
git lfs install
git clone https://huggingface.co/datasets/wanglab/cafa5

cp af_temp_holdout_2022_2025_shards/* ../../cafa5/structures_temp_holdout_2022_2025/


git add structures_temp_holdout_2022_2025/
git commit -m "Add AlphaFold structures for temp holdout 2022 to 2025 proteins"
git push

# Adding Structure Path column

In [35]:
import os

# --- Configuration ---
ACCESSION_FILE = "temporal-holdout-ext-proteins.txt"
DIR_ALPHAFOLD = "alphafold_structures"

print("Starting structure check...")

try:
    # 1. Load accession file
    with open(ACCESSION_FILE, 'r') as f:
        all_accessions = [line.strip() for line in f if line.strip()]
except FileNotFoundError:
    print(f"Error: Accession file '{ACCESSION_FILE}' not found.")
    exit()
except Exception as e:
    print(f"An error occurred while reading the accession file: {e}")
    exit()

accessions_with_structures = []
accessions_without_structures = []

for accession in all_accessions:
    pdb_path = os.path.join(DIR_ALPHAFOLD, f"AF-{accession}-F1-model_v6.pdb")
    if os.path.exists(pdb_path):
        accessions_with_structures.append(accession)
    else:
        accessions_without_structures.append(accession)

print("\n--- Structure Existence Report ---")
print(f"Total accessions loaded from '{ACCESSION_FILE}': {len(all_accessions)}")
print(f"Directory checked: '{DIR_ALPHAFOLD}'")
print("-" * 40)
print(f"Accessions with a structure found: {len(accessions_with_structures)}")
print(f"Accessions with NO structure found: {len(accessions_without_structures)}")
print("-" * 40)

# Optionally, print out the lists, comment out if not needed:
# print("WITH STRUCTURE:", accessions_with_structures)
# print("WITHOUT STRUCTURE:", accessions_without_structures)


Starting structure check...

--- Structure Existence Report ---
Total accessions loaded from 'temporal-holdout-ext-proteins.txt': 17812
Directory checked: 'alphafold_structures'
----------------------------------------
Accessions with a structure found: 16081
Accessions with NO structure found: 1731
----------------------------------------


In [36]:
final_temporal_holdout_ext['structure_path'] = final_temporal_holdout_ext['protein_id'].apply(
    lambda x: f"AF-{x}-F1-model_v6.pdb" if x in accessions_with_structures else None
)

In [38]:
final_temporal_holdout_ext['structure_path'].isna().sum()

1731

# Getting InterPro Domains for the Proteins

In [ ]:
from datasets import load_dataset
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
from tqdm import tqdm

def fetch_one_protein(accession, max_retries=3):
    url = f"https://www.ebi.ac.uk/interpro/api/entry/InterPro/protein/UniProt/{accession}"
    headers = {"Accept": "application/json"}

    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 204:
                return []
            r.raise_for_status()

            results = r.json().get("results", [])
            rows = []

            for item in results:
                ipr_id = item.get("metadata", {}).get("accession")
                entry_name = item.get("metadata", {}).get("name")
                ipr_type = item.get("metadata", {}).get("type")

                all_fragments = []
                for loc in item.get("proteins", []):  # fixed: must go through item['proteins']
                    for entry in loc.get("entry_protein_locations", []):
                        all_fragments.extend(entry.get("fragments", []))

                if all_fragments:
                    start = min(f["start"] for f in all_fragments if "start" in f)
                    end = max(f["end"] for f in all_fragments if "end" in f)

                    rows.append({
                        "accession": accession,
                        "interpro_id": ipr_id,
                        "entry_name": entry_name,
                        "type": ipr_type,
                        "start": start,
                        "end": end,
                        "n_fragments": len(all_fragments)
                    })
            return rows

        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Failed for {accession}: {e}")
            sleep(2 * (attempt + 1))
    return []

def fetch_all_interpro_parallel(
    all_accessions,
    output_file="interpro_domains.tsv",
    interpro_lookup_file="interpro_lookup.tsv",
    max_threads=32
):
    protein_rows = []
    interpro_lookup = {}

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(fetch_one_protein, acc): acc for acc in all_accessions}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching InterPro"):
            rows = future.result()
            protein_rows.extend(rows)

    # Extract lookup table
    for row in protein_rows:
        ipr = row["interpro_id"]
        if ipr not in interpro_lookup:
            interpro_lookup[ipr] = {
                "entry_name": row["entry_name"],
                "type": row["type"]
            }

    # Save data
    pd.DataFrame(protein_rows).to_csv(output_file, sep="\t", index=False)
    pd.DataFrame([
        {"interpro_id": ipr, "entry_name": val["entry_name"], "type": val["type"]}
        for ipr, val in interpro_lookup.items()
    ]).to_csv(interpro_lookup_file, sep="\t", index=False)

    print(f"✅ Done. Fetched {len(protein_rows)} entries across {len(all_accessions)} proteins")

# ---------- Main execution ----------
if __name__ == "__main__":
    accessions_path = "temporal-holdout-ext-proteins.txt"

    print(f"Reading accessions from {accessions_path}")
    protein_accessions = list(pd.read_csv(accessions_path, header=None)[0])  # your query list
    print(f"Found {len(protein_accessions)} accessions")

    fetch_all_interpro_parallel(
        list(protein_accessions),
        output_file="temporal-holdout-ext-interpro-domains.tsv",
        interpro_lookup_file="temporal-holdout-ext-interpro-lookup.tsv",
        max_threads=32
    )

In [39]:
interpro = pd.read_csv('temporal-holdout-ext-interpro-domains.tsv', sep='\t')

In [40]:
interpro

,accession,interpro_id,entry_name,type,start,end,n_fragments
0,A0A024R531,IPR001523,Paired domain,domain,9,135,1
1,A0A024R531,IPR009057,Homedomain-like superfamily,homologous_superfamily,11,133,1
2,A0A024R531,IPR022130,Paired-box protein 2 C-terminal,domain,282,386,1
3,A0A024R531,IPR036388,Winged helix-like DNA-binding domain superfamily,homologous_superfamily,1,141,2
4,A0A024R531,IPR043182,Paired DNA-binding domain,domain,43,59,1
...,...,...,...,...,...,...,...
85523,W8DXL4,IPR013783,Immunoglobulin-like fold,homologous_superfamily,256,565,2
85524,W8DXL4,IPR032675,Leucine-rich repeat domain superfamily,homologous_superfamily,18,255,1
85525,W8DXL4,IPR036116,Fibronectin type III superfamily,homologous_superfamily,490,558,1
85526,W8DXL4,IPR036179,Immunoglobulin-like domain superfamily,homologous_superfamily,252,346,1


In [41]:
set(interpro['type'])

{'active_site',
 'binding_site',
 'conserved_site',
 'domain',
 'family',
 'homologous_superfamily',
 'ptm',
 'repeat'}

In [42]:
from collections import defaultdict

# Define your custom type priority
type_order = {
    'homologous_superfamily': 0,
    'family': 1,
    'domain': 2,
    'conserved_site': 3,
    'ptm': 4,
    'repeat': 5,
    'active_site': 6,
    'binding_site': 7
}

# Intermediate mapping
accession_to_entries = defaultdict(list)

# Store everything per accession
for acc, ipr, start, end, typ in zip(
    interpro["accession"], interpro["interpro_id"],
    interpro["start"], interpro["end"], interpro["type"]
):
    accession_to_entries[acc].append((ipr, start, end, typ))

# Final output structure
collapsed_data = []

for acc, entries in accession_to_entries.items():
    interpro_ids = set()
    interpro_location = {}
    
    # Sort by (type priority, start)
    entries_sorted = sorted(
        entries,
        key=lambda x: (type_order.get(x[3], 99), x[1])  # (type, start)
    )

    for ipr, start, end, typ in entries_sorted:
        if ipr in interpro_location:
            raise ValueError(f"InterPro ID {ipr} appears more than once for protein {acc}")
        interpro_ids.add(ipr)
        interpro_location[ipr] = (start, end)

    collapsed_data.append({
        "accession": acc,
        "interpro_ids": list(interpro_ids),
        "interpro_location": interpro_location
    })

# Convert to DataFrame
collapsed_df = pd.DataFrame(collapsed_data)

In [43]:
collapsed_df

,accession,interpro_ids,interpro_location
0,A0A024R531,"[IPR043182, IPR009057, IPR036388, IPR001523, I...","{'IPR036388': (1, 141), 'IPR009057': (11, 133)..."
1,A0A080WMA9,"[IPR036291, IPR005676, IPR000319, IPR012280, I...","{'IPR036291': (8, 182), 'IPR051823': (6, 361),..."
2,A0A087WQN9,"[IPR002035, IPR036465]","{'IPR036465': (20, 74), 'IPR002035': (35, 74)}"
3,A0A087WT78,"[IPR001404, IPR020568, IPR036890, IPR037196, I...","{'IPR036890': (66, 358), 'IPR020568': (339, 59..."
4,A0A087WS16,"[IPR050525, IPR002035, IPR002223, IPR041900, I...","{'IPR036465': (15, 2200), 'IPR013783': (2498, ..."
...,...,...,...
17007,Z4YIA7,"[IPR002048, IPR011992, IPR018247]","{'IPR011992': (68, 311), 'IPR002048': (70, 302..."
17008,V9HW26,"[IPR020003, IPR038376, IPR004100, IPR036121, I...","{'IPR023366': (44, 136), 'IPR036121': (53, 137..."
17009,X5JA13,"[IPR009976, IPR048627, IPR048625]","{'IPR009976': (20, 812), 'IPR048625': (90, 206..."
17010,X1WHV8,"[IPR014001, IPR027417, IPR048333, IPR007502, I...","{'IPR027417': (3, 554), 'IPR014001': (61, 227)..."


In [44]:
temp_hold_protein_ids = set(final_temporal_holdout_ext['protein_id'])

accessions_not_in_test = [
    c for c in temp_hold_protein_ids
    if c not in set(collapsed_df['accession'])
]

In [45]:
len(accessions_not_in_test)

800

# Update Train and Test with Interpro Data

In [46]:
interpro_collapsed = collapsed_df.rename(columns={"accession": "protein_id"})

final_temporal_holdout_ext = final_temporal_holdout_ext.merge(interpro_collapsed, on="protein_id", how="left")

In [47]:
print("Rows where test interpro is None:", final_temporal_holdout_ext["interpro_ids"].isnull().sum())

Rows where test interpro is None: 800


## Interpro Metadata

In [48]:
interpro_metadata = pd.read_csv('temporal-holdout-ext-interpro-lookup.tsv', sep='\t')

In [49]:
interpro_metadata

,interpro_id,entry_name,type
0,IPR001523,Paired domain,domain
1,IPR009057,Homedomain-like superfamily,homologous_superfamily
2,IPR022130,Paired-box protein 2 C-terminal,domain
3,IPR036388,Winged helix-like DNA-binding domain superfamily,homologous_superfamily
4,IPR043182,Paired DNA-binding domain,domain
...,...,...,...
14181,IPR000332,Beta 2 adrenoceptor,family
14182,IPR002233,Adrenoceptor family,family
14183,IPR009976,Exocyst complex component Sec10-like,family
14184,IPR048625,"Exocyst complex component Sec10, N-terminal",domain


**QC to check if every interpro ID in interlabel reasoning is in this metadata df**

In [50]:
# Step 1: create a set of valid InterPro IDs
valid_ids = set(interpro_metadata["interpro_id"].dropna())

# Step 2: explode interpro_ids into separate rows
exploded = final_temporal_holdout_ext[["interpro_ids"]].explode("interpro_ids")

# Step 3: filter out null values first
exploded = exploded[exploded["interpro_ids"].notna()]

# Step 4: mark invalid IDs
exploded["is_invalid"] = ~exploded["interpro_ids"].isin(valid_ids)

# Step 5: count invalids
invalid_count = exploded["is_invalid"].sum()

print(invalid_count if invalid_count > 0 else 0)

0


**All interpro ids in interlabel dataframes are in this metadata df**

# Upload Data

In [51]:
import json

In [53]:
#Making it into a json object so that huggingface can handle it
final_temporal_holdout_ext["interpro_location"] = final_temporal_holdout_ext["interpro_location"].apply(json.dumps)

In [54]:
from datasets import Dataset

test_hf = Dataset.from_pandas(final_temporal_holdout_ext, preserve_index=False)

In [55]:
from datasets import DatasetDict

temp_hold_dataset = DatasetDict({
    "test": test_hf
})

In [56]:
temp_hold_dataset

DatasetDict({
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'go_bp_created', 'go_mf_created', 'go_cc_created', 'date_added', 'last_modified', 'length', 'structure_path', 'interpro_ids', 'interpro_location'],
        num_rows: 17812
    })
})

In [57]:
temp_hold_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="temporal_holdout_ext_2022_2025_reasoning",
    commit_message="Upload Temporal holdout data with new reasoning data"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/18 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   3%|3         |  547kB / 17.4MB            

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/05be3da2e39c65d4fd179af5f2d07529278dd3b3', commit_message='Upload Temporal holdout data with new reasoning data', commit_description='', oid='05be3da2e39c65d4fd179af5f2d07529278dd3b3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

In [58]:
from datasets import Dataset

# Ensure the columns that contain dictionaries or lists are JSON-serializable
interpro_hf = Dataset.from_pandas(interpro_metadata, preserve_index=False)

In [59]:
from datasets import DatasetDict

interlabel_dataset = DatasetDict({
    "metadata": interpro_hf,
})

In [60]:
interlabel_dataset

DatasetDict({
    metadata: Dataset({
        features: ['interpro_id', 'entry_name', 'type'],
        num_rows: 14186
    })
})

In [61]:
interlabel_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="interpro_temp_holdout_ext_22_25_metadata",
    commit_message="Upload the InterPro temporal holdout 2022 to 2025 Metadata"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/15 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########|  395kB /  395kB            

                                        : 100%|##########|  395kB /  395kB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/399953d84085afceba2ee2fc0a046ffbbb2f8479', commit_message='Upload the InterPro temporal holdout 2022 to 2025 Metadata', commit_description='', oid='399953d84085afceba2ee2fc0a046ffbbb2f8479', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)